In [1]:
import pandas as pd

raw = pd.read_csv(
    "data_cleaning/data/medi-cal.csv",
    low_memory=False
)

df = raw.copy()

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 372778
Columns: 46


,_id,OBJECTID,Provider_Number,NPI,Owner_Number,Service_Location_Number,Legal_Name,Enroll_Status_Eff_DT,Provider_Taxonomy,ANC_Provider_Type,...,Acc_Interior_Building,Acc_Parking,Acc_Restroom,Acc_Medical_Equipment,Acc_Patient_Areas,Acc_Patient_Diagnostic,Provider_Source,Latitude,Longitude,CountyName
0,1,1,1003000639,1.003001e+09,1,1,"BENHARASH, PEYMAN MD",2011-01-19 0:00:00,No Taxonomy Provided,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,34.063722,-118.443467,Los Angeles County
1,2,2,1003000563,1.003001e+09,1,1,"CHUN, RICHARD B D MD INC",2002-04-01 0:00:00,207RG0100X,Gastroenterology,...,NaN,NaN,NaN,NaN,NaN,NaN,1,38.734427,-121.271995,Placer County
2,3,3,1003000563,1.003001e+09,1,1,"CHUN, RICHARD B D MD INC",2002-04-01 0:00:00,207RG0100X,Gastroenterology,...,NaN,NaN,NaN,NaN,NaN,NaN,1,38.734427,-121.271995,Placer County
3,4,4,1003001157,1.003001e+09,1,1,"KIM, DAVID C MD",2003-07-18 0:00:00,No Taxonomy Provided,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,34.099394,-117.421262,San Bernardino County
4,5,5,1003000225,1.003000e+09,1,1,"MORGAN, ANNA C A MD",2008-10-15 0:00:00,No Taxonomy Provided,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,37.891776,-122.058050,Contra Costa County


In [2]:

# Inspect every column
column_check = pd.DataFrame({
    "data_type": df.dtypes,
    "missing_count": df.isna().sum(),
    "unique_values": df.nunique()
})

column_check

,data_type,missing_count,unique_values
_id,int64,0,372778
OBJECTID,int64,0,372778
Provider_Number,str,0,302802
NPI,float64,49,302753
Owner_Number,int64,0,6
Service_Location_Number,int64,0,189
Legal_Name,str,0,299795
Enroll_Status_Eff_DT,str,0,12399
Provider_Taxonomy,str,0,613
ANC_Provider_Type,str,213269,39


In [3]:
df.columns.tolist()

['_id',
 'OBJECTID',
 'Provider_Number',
 'NPI',
 'Owner_Number',
 'Service_Location_Number',
 'Legal_Name',
 'Enroll_Status_Eff_DT',
 'Provider_Taxonomy',
 'ANC_Provider_Type',
 'NEMT_NMT_Provider_Type',
 'FI_Provider_Type_CD',
 'FI_Provider_Type',
 'Provider_License',
 'FI_Provider_Specialty_CD',
 'FI_Provider_Specialty',
 'Out_of_State_Indicator',
 'In_Out_State',
 'Address_Attention',
 'Address',
 'Address2',
 'City',
 'State',
 'ZIP',
 'ZIP_4',
 'DHCS_County_CD',
 'FIPS_County_CD',
 'Phone_Number',
 'Medicaid_Patients',
 'CHIP_Patients',
 'Telehealth_Services',
 'Provider_Website',
 'Threshold_Languages',
 'Other_Languages',
 'Acc_Exam_Room',
 'Acc_Exterior_Building',
 'Acc_Interior_Building',
 'Acc_Parking',
 'Acc_Restroom',
 'Acc_Medical_Equipment',
 'Acc_Patient_Areas',
 'Acc_Patient_Diagnostic',
 'Provider_Source',
 'Latitude',
 'Longitude',
 'CountyName']

In [4]:
# Check exact duplicate rows
exact_duplicates = df.duplicated().sum()

# Check repeated provider-location identifiers
id_columns = [
    "Provider_Number",
    "Owner_Number",
    "Service_Location_Number"
]

repeated_ids = df.duplicated(
    subset=id_columns,
    keep=False
).sum()

print("Exact duplicate rows:", exact_duplicates)
print("Rows with repeated provider-location IDs:", repeated_ids)

Exact duplicate rows: 0
Rows with repeated provider-location IDs: 71084


In [5]:
# View examples of repeated provider-location IDs
repeated_records = (
    df[df.duplicated(subset=id_columns, keep=False)]
    .sort_values(id_columns)
)

repeated_records[
    id_columns
    + [
        "NPI",
        "Legal_Name",
        "Enroll_Status_Eff_DT",
        "Provider_Taxonomy",
        "FI_Provider_Specialty",
        "Address",
        "City",
        "ZIP"
    ]
].head(20)

,Provider_Number,Owner_Number,Service_Location_Number,NPI,Legal_Name,Enroll_Status_Eff_DT,Provider_Taxonomy,FI_Provider_Specialty,Address,City,ZIP
1,1003000563,1,1,1.003001e+09,"CHUN, RICHARD B D MD INC",2002-04-01 0:00:00,207RG0100X,Gastroenterology (M.D. only),729 SUNRISE AVE,ROSEVILLE,95661.0
2,1003000563,1,1,1.003001e+09,"CHUN, RICHARD B D MD INC",2002-04-01 0:00:00,207RG0100X,Internal Medicine,729 SUNRISE AVE,ROSEVILLE,95661.0
22,1003008855,0,0,1.003009e+09,"LEE, ALISIA PA-C",2025-07-21 0:00:00,Taxonomy not applicable,NaN,31180 ROAD 72,VISALIA,93291.0
23,1003008855,0,0,1.003009e+09,"LEE, ALISIA D",2016-03-01 0:00:00,Taxonomy not applicable,NaN,2112 COURTLEIGH DRIVE,BAKERSFIELD,93309.0
37,1003012733,1,1,1.003013e+09,"GANNON, LAUREN C MD",2014-02-07 0:00:00,No Taxonomy Provided,Internal Medicine,5454 EL CAJON BLVD,SAN DIEGO,92115.0
39,1003012733,1,1,1.003013e+09,"GANNON, LAUREN C MD",2014-02-07 0:00:00,No Taxonomy Provided,Pediatrics,5454 EL CAJON BLVD,SAN DIEGO,92115.0
41,1003013269,0,0,1.003013e+09,"CHELSON, JILLIAN M PA-C",2022-08-10 0:00:00,Taxonomy not applicable,NaN,2018 MISSION ST,SANTA CRUZ,95060.0
42,1003013269,0,0,1.003013e+09,"CHELSON, JILLIAN M PA",2021-05-23 0:00:00,Taxonomy not applicable,NaN,21507 EAST CLIFF DRIVE,SANTA CRUZ,95061.0
44,1003013319,1,1,1.003013e+09,"ORSBORN, JONATHAN W MD",2018-01-01 0:00:00,No Taxonomy Provided,Pediatrics,13199 E MONTVIEW BLVD,AURORA,80045.0
45,1003013319,1,1,1.003013e+09,"ORSBORN, JONATHAN W MD",2018-01-01 0:00:00,No Taxonomy Provided,Emergency Medicine (Urgent Care),13199 E MONTVIEW BLVD,AURORA,80045.0


In [6]:
# Save important original values for before/after checks
before_cleaning = df.copy()

# 1. Clean spaces in all text columns
text_columns = df.select_dtypes(include=["object", "string"]).columns

for column in text_columns:
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# Convert blank text into missing values
df[text_columns] = df[text_columns].replace("", pd.NA)


# 2. Clean ID columns and preserve them as text
# Keep Provider_Number as text
df["Provider_Number"] = (
    df["Provider_Number"]
    .astype("string")
    .str.strip()
)

# Clean the other identifier columns
numeric_identifier_columns = [
    "NPI",
    "Owner_Number",
    "Service_Location_Number"
]

for column in numeric_identifier_columns:
    df[column] = (
        pd.to_numeric(df[column], errors="coerce")
        .astype("Int64")
        .astype("string")
    )

# 3. Clean ZIP as a five-character identifier
df["ZIP"] = (
    pd.to_numeric(df["ZIP"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str.zfill(5)
)


# 4. Convert the enrollment date into a real date
df["Enroll_Status_Eff_DT"] = pd.to_datetime(
    df["Enroll_Status_Eff_DT"],
    errors="coerce"
)


# 5. Standardize state and county text
df["State"] = df["State"].str.upper()
df["CountyName"] = df["CountyName"].str.title()


print("Basic cleaning completed.")
print("Rows after cleaning:", len(df))
print("No rows have been removed.")

Basic cleaning completed.
Rows after cleaning: 372778
No rows have been removed.


In [7]:
# Check failed date conversions
original_dates_present = before_cleaning["Enroll_Status_Eff_DT"].notna()
failed_date_conversions = (
    original_dates_present &
    df["Enroll_Status_Eff_DT"].isna()
).sum()

# Check ZIP codes
invalid_zip_count = (
    df["ZIP"].notna() &
    ~df["ZIP"].str.fullmatch(r"\d{5}", na=False)
).sum()

# Check impossible latitude and longitude values
invalid_latitude_count = (
    df["Latitude"].notna() &
    ~df["Latitude"].between(-90, 90)
).sum()

invalid_longitude_count = (
    df["Longitude"].notna() &
    ~df["Longitude"].between(-180, 180)
).sum()

# Check future enrollment dates
future_date_count = (
    df["Enroll_Status_Eff_DT"] >
    pd.Timestamp.today().normalize()
).sum()

print("Failed date conversions:", failed_date_conversions)
print("Invalid ZIP codes:", invalid_zip_count)
print("Invalid latitudes:", invalid_latitude_count)
print("Invalid longitudes:", invalid_longitude_count)
print("Future enrollment dates:", future_date_count)

print("\nEnrollment date range:")
print(df["Enroll_Status_Eff_DT"].min())
print(df["Enroll_Status_Eff_DT"].max())

print("\nMost common states:")
print(df["State"].value_counts(dropna=False).head(10))

Failed date conversions: 0
Invalid ZIP codes: 0
Invalid latitudes: 0
Invalid longitudes: 0
Future enrollment dates: 0

Enrollment date range:
1966-01-01 00:00:00
2026-09-04 00:00:00

Most common states:
State
CA      361223
NV        3075
<NA>      2106
AZ        1313
OR         672
TX         667
CO         501
OH         203
FL         180
ID         177
Name: count, dtype: int64[pyarrow]


In [8]:
# Compare missing values before and after cleaning
missing_before = before_cleaning.isna().sum()
missing_after = df.isna().sum()

missing_check = pd.DataFrame({
    "missing_before": missing_before,
    "missing_after": missing_after,
    "change": missing_after - missing_before
})

missing_check = missing_check[
    (missing_check["missing_before"] > 0) |
    (missing_check["missing_after"] > 0)
]

missing_check

,missing_before,missing_after,change
NPI,49,49,0
ANC_Provider_Type,213269,213269,0
NEMT_NMT_Provider_Type,371410,371410,0
FI_Provider_Type_CD,26744,26744,0
FI_Provider_Type,26744,26744,0
Provider_License,36375,36375,0
FI_Provider_Specialty_CD,150217,150217,0
FI_Provider_Specialty,150217,150217,0
Out_of_State_Indicator,62254,62254,0
In_Out_State,62254,62254,0


In [9]:
# Save the full cleaned dataset
df.to_csv("cleaned_medi_cal.csv", index=False)

# Select up to 50 reproducible sample rows
cleaned_sample = df.sample(
    n=min(50, len(df)),
    random_state=576
)

cleaned_sample.to_csv(
    "cleaned_medi_cal_sample.csv",
    index=False
)

print("Full cleaned rows saved:", len(df))
print("Sample rows saved:", len(cleaned_sample))

Full cleaned rows saved: 372778
Sample rows saved: 50


In [10]:
# ------------------------------------------------------------
# FIVE INDIVIDUAL RECORD CHECKS
# ------------------------------------------------------------

# 1. A repeated provider-location ID
repeated_index = repeated_records.index[0]

# 2. A record with a missing state
missing_state_index = df[df["State"].isna()].index[0]

# 3. An out-of-state provider
out_of_state_index = df[
    df["State"].notna() & (df["State"] != "CA")
].index[0]

# 4. The record with the earliest enrollment date
earliest_date_index = df["Enroll_Status_Eff_DT"].idxmin()

# 5. A standard California record
standard_index = df[
    (df["State"] == "CA") &
    df["ZIP"].notna() &
    df["CountyName"].notna()
].index[0]

check_indices = [
    repeated_index,
    missing_state_index,
    out_of_state_index,
    earliest_date_index,
    standard_index
]

record_checks = pd.DataFrame({
    "check": [
        "Repeated provider-location ID",
        "Missing state",
        "Out-of-state provider",
        "Earliest enrollment date",
        "Standard California record"
    ],
    "reason": [
        "Repeated IDs may contain different valid enrollment or specialty information",
        "Missing state should remain missing instead of being invented",
        "Out-of-state providers may be valid and should not automatically be removed",
        "An old date is unusual but not automatically incorrect",
        "A normal record should remain available after cleaning"
    ],
    "original_value": [
        str(before_cleaning.loc[repeated_index, id_columns].to_dict()),
        before_cleaning.loc[missing_state_index, "State"],
        before_cleaning.loc[out_of_state_index, "State"],
        before_cleaning.loc[earliest_date_index, "Enroll_Status_Eff_DT"],
        before_cleaning.loc[standard_index, "ZIP"]
    ],
    "expected_result": [
        "Keep the record",
        "Keep as missing",
        "Keep the record",
        "Keep valid historical date",
        "Five-character ZIP"
    ],
    "actual_result": [
        "Record kept" if repeated_index in df.index else "Record removed",
        df.loc[missing_state_index, "State"],
        df.loc[out_of_state_index, "State"],
        df.loc[earliest_date_index, "Enroll_Status_Eff_DT"],
        df.loc[standard_index, "ZIP"]
    ],
    "matches_expected": [
        repeated_index in df.index,
        pd.isna(df.loc[missing_state_index, "State"]),
        out_of_state_index in df.index,
        pd.notna(df.loc[earliest_date_index, "Enroll_Status_Eff_DT"]),
        len(df.loc[standard_index, "ZIP"]) == 5
    ]
})

record_checks

,check,reason,original_value,expected_result,actual_result,matches_expected
0,Repeated provider-location ID,Repeated IDs may contain different valid enrol...,"{'Provider_Number': '1003000563', 'Owner_Numbe...",Keep the record,Record kept,True
1,Missing state,Missing state should remain missing instead of...,NaN,Keep as missing,<NA>,True
2,Out-of-state provider,Out-of-state providers may be valid and should...,NV,Keep the record,NV,True
3,Earliest enrollment date,An old date is unusual but not automatically i...,1966-01-01 0:00:00,Keep valid historical date,1966-01-01 00:00:00,True
4,Standard California record,A normal record should remain available after ...,90095.0,Five-character ZIP,90095,True


In [11]:
# Include the five checked records, then add 45 reproducible random records
difficult_rows = df.loc[check_indices]

remaining_rows = df.drop(index=check_indices).sample(
    n=45,
    random_state=576
)

cleaned_sample = pd.concat(
    [difficult_rows, remaining_rows],
    ignore_index=True
)

cleaned_sample.to_csv(
    "cleaned_medi_cal_sample.csv",
    index=False
)

record_checks.to_csv(
    "five_record_checks.csv",
    index=False
)

print("Five record checks saved.")
print("Improved 50-row sample saved.")

Five record checks saved.
Improved 50-row sample saved.


In [12]:
cleaning_decision_log = pd.DataFrame({
    "cleaning_rule": [
        "Trim leading, trailing, and repeated spaces in text columns",
        "Convert blank strings to missing values",
        "Store provider identifiers as text",
        "Store ZIP codes as five-character text",
        "Convert enrollment dates to datetime",
        "Standardize state abbreviations to uppercase",
        "Standardize county names to title case",
        "Keep repeated provider-location IDs",
        "Keep missing values",
        "Keep valid out-of-state providers"
    ],
    "reason": [
        "Prevents one value from appearing as multiple categories because of spacing",
        "Blank strings and missing values should be represented consistently",
        "Identifiers are labels and should not be used in calculations",
        "ZIP codes are identifiers and leading zeros are meaningful",
        "Allows dates to be checked and analyzed correctly",
        "Prevents values such as ca and CA from becoming separate categories",
        "Makes county categories consistent",
        "Repeated IDs may represent different specialties or enrollment records",
        "Missing information should not be invented",
        "The dataset legitimately contains enrolled providers outside California"
    ],
    "verification_check": [
        "Compared text before and after cleaning",
        "Compared missing counts before and after cleaning",
        "Checked identifier data types after conversion",
        "Confirmed every nonmissing ZIP has five digits",
        "Confirmed zero failed date conversions",
        "Reviewed state value counts",
        "Reviewed county values",
        "Found 71,084 repeated-ID rows and inspected examples",
        "Reported missing counts without dropping rows",
        "Reviewed out-of-state records and kept them"
    ]
})

cleaning_decision_log.to_csv(
    "cleaning_decision_log.csv",
    index=False
)

cleaning_decision_log

,cleaning_rule,reason,verification_check
0,"Trim leading, trailing, and repeated spaces in...",Prevents one value from appearing as multiple ...,Compared text before and after cleaning
1,Convert blank strings to missing values,Blank strings and missing values should be rep...,Compared missing counts before and after cleaning
2,Store provider identifiers as text,Identifiers are labels and should not be used ...,Checked identifier data types after conversion
3,Store ZIP codes as five-character text,ZIP codes are identifiers and leading zeros ar...,Confirmed every nonmissing ZIP has five digits
4,Convert enrollment dates to datetime,Allows dates to be checked and analyzed correctly,Confirmed zero failed date conversions
5,Standardize state abbreviations to uppercase,Prevents values such as ca and CA from becomin...,Reviewed state value counts
6,Standardize county names to title case,Makes county categories consistent,Reviewed county values
7,Keep repeated provider-location IDs,Repeated IDs may represent different specialti...,"Found 71,084 repeated-ID rows and inspected ex..."
8,Keep missing values,Missing information should not be invented,Reported missing counts without dropping rows
9,Keep valid out-of-state providers,The dataset legitimately contains enrolled pro...,Reviewed out-of-state records and kept them


In [13]:
# Final reconciliation
starting_rows = len(raw)
final_rows = len(df)
rows_removed = starting_rows - final_rows

# Main project metrics
provider_location_records = len(df)
unique_providers = df["Provider_Number"].nunique(dropna=True)
records_with_county = df["CountyName"].notna().sum()
records_missing_county = df["CountyName"].isna().sum()

verification_summary = pd.DataFrame({
    "check": [
        "Starting rows",
        "Final rows",
        "Rows removed",
        "Exact duplicates",
        "Rows with repeated provider-location IDs",
        "Provider-location records",
        "Unique providers",
        "Records included in county analysis",
        "Records excluded from county analysis because county is missing"
    ],
    "result": [
        starting_rows,
        final_rows,
        rows_removed,
        exact_duplicates,
        repeated_ids,
        provider_location_records,
        unique_providers,
        records_with_county,
        records_missing_county
    ]
})

verification_summary.to_csv(
    "verification_summary.csv",
    index=False
)

verification_summary

,check,result
0,Starting rows,372778
1,Final rows,372778
2,Rows removed,0
3,Exact duplicates,0
4,Rows with repeated provider-location IDs,71084
5,Provider-location records,372778
6,Unique providers,302802
7,Records included in county analysis,370321
8,Records excluded from county analysis because ...,2457


In [14]:
print("Missing county count:", df["CountyName"].isna().sum())
print(
    "Records with county:",
    df["CountyName"].notna().sum()
)
print(
    "Unique provider count:",
    df["Provider_Number"].nunique(dropna=True)
)

Missing county count: 2457
Records with county: 370321
Unique provider count: 302802


In [15]:
# Display the five record checks as a readable table

record_check_table = record_checks.rename(columns={
    "check": "Record checked",
    "original_value": "Original value",
    "expected_result": "Expected result",
    "actual_result": "Actual result",
    "matches_expected": "Matched?"
}).copy()

record_check_table["Original value"] = (
    record_check_table["Original value"]
    .fillna("Missing (NaN)")
)

record_check_table["Actual result"] = (
    record_check_table["Actual result"]
    .fillna("Missing (<NA>)")
)

display(
    record_check_table.style
    .set_properties(**{
        "text-align": "left",
        "white-space": "normal"
    })
    .set_table_styles([{
        "selector": "th",
        "props": [
            ("text-align", "left"),
            ("background-color", "#dddddd")
        ]
    }])
    .hide(axis="index")
)

Record checked,reason,Original value,Expected result,Actual result,Matched?
Repeated provider-location ID,Repeated IDs may contain different valid enrollment or specialty information,"{'Provider_Number': '1003000563', 'Owner_Number': 1, 'Service_Location_Number': 1}",Keep the record,Record kept,True
Missing state,Missing state should remain missing instead of being invented,Missing (NaN),Keep as missing,Missing (),True
Out-of-state provider,Out-of-state providers may be valid and should not automatically be removed,NV,Keep the record,NV,True
Earliest enrollment date,An old date is unusual but not automatically incorrect,1966-01-01 0:00:00,Keep valid historical date,1966-01-01 00:00:00,True
Standard California record,A normal record should remain available after cleaning,90095.000000,Five-character ZIP,90095,True
